# 从零构建AI推理大模型 - BASIC编程代码生成

## 项目概述
- **功能**：输入自然语言英文提示语（一位数加减乘除），输出BASIC程序
- **示例**：`"Input 3 and 4, and print their sum"` → `10 INPUT A\n20 INPUT B\n30 PRINT A + B`
- **平台**：Kaggle 2×T4 GPU, 30GB RAM
- **架构**：从零训练小型GPT (Decoder-Only Transformer), ~15M参数

## 开发流程
1. 数据集生成
2. 分词器训练
3. 模型构建
4. 模型训练
5. 推理验证

## Step 0: 环境检查与配置

In [ ]:
import os
import json
import time
import math
import gc
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

In [ ]:
class Config:
    data_dir = "./data"
    data_path = os.path.join(data_dir, "dataset.json")
    tokenizer_path = os.path.join(data_dir, "tokenizer.json")
    model_dir = "./checkpoints"
    model_path = os.path.join(model_dir, "model.pt")

    num_samples = 50000
    train_ratio = 0.95
    seed = 42

    vocab_size = 2048
    max_seq_len = 128

    d_model = 256
    n_heads = 8
    n_layers = 6
    d_ff = 1024
    dropout = 0.1

    batch_size = 64
    learning_rate = 3e-4
    weight_decay = 0.01
    max_epochs = 30
    warmup_steps = 500
    grad_clip = 1.0

    num_workers = 2
    num_gpus = 2

    inference_max_len = 128
    inference_temperature = 0.8
    inference_top_k = 50

    special_tokens = ["<pad>", "<eos>", "<sep>", "<unk>"]
    pad_id = 0
    eos_id = 1
    sep_id = 2
    unk_id = 3

config = Config()
print("Configuration loaded.")
print(f"  Model: d_model={config.d_model}, n_heads={config.n_heads}, n_layers={config.n_layers}, d_ff={config.d_ff}")
print(f"  Training: batch_size={config.batch_size}, lr={config.learning_rate}, epochs={config.max_epochs}")
print(f"  Data: {config.num_samples} samples, max_seq_len={config.max_seq_len}")

## Step 1: 数据集生成

程序化生成50,000条配对数据：
- 加法(add): ~12,500条
- 减法(sub): ~12,500条
- 乘法(mul): ~12,500条
- 除法(div): ~12,500条

每种运算使用15种不同的英文表达模板，数字范围0-9。

In [ ]:
ADD_TEMPLATES = [
    "Input {a} and {b}, and print their sum",
    "Calculate the sum of {a} and {b}",
    "Add {a} and {b} together and print the result",
    "Write a program to add {a} and {b}",
    "Find the sum of {a} and {b}",
    "Compute {a} plus {b}",
    "What is {a} added to {b}?",
    "Print the result of adding {a} and {b}",
    "Add together {a} and {b} and display it",
    "Show the sum when you add {a} and {b}",
    "Calculate {a} + {b}",
    "Find the total of {a} and {b}",
    "Sum up {a} and {b} and print",
    "Get the addition result of {a} and {b}",
    "What do you get when you add {a} and {b}?",
]

SUB_TEMPLATES = [
    "Input {a} and {b}, and print their difference",
    "Calculate the difference of {a} and {b}",
    "Subtract {b} from {a} and print the result",
    "Write a program to subtract {b} from {a}",
    "Find the difference between {a} and {b}",
    "Compute {a} minus {b}",
    "What is {a} subtracted by {b}?",
    "Print the result of subtracting {b} from {a}",
    "Show the difference when you subtract {b} from {a}",
    "Calculate {a} - {b}",
    "Subtract {b} from {a} and display the answer",
    "What is the result of {a} minus {b}?",
    "Find {a} take away {b}",
    "Get the subtraction result of {a} and {b}",
    "What do you get when you subtract {b} from {a}?",
]

MUL_TEMPLATES = [
    "Input {a} and {b}, and print their product",
    "Calculate the product of {a} and {b}",
    "Multiply {a} and {b} together and print the result",
    "Write a program to multiply {a} and {b}",
    "Find the product of {a} and {b}",
    "Compute {a} times {b}",
    "What is {a} multiplied by {b}?",
    "Print the result of multiplying {a} and {b}",
    "Show the product when you multiply {a} and {b}",
    "Calculate {a} * {b}",
    "Multiply {a} by {b} and display the answer",
    "What is the result of {a} times {b}?",
    "Find the multiplication of {a} and {b}",
    "Get the product of {a} and {b}",
    "What do you get when you multiply {a} and {b}?",
]

DIV_TEMPLATES = [
    "Input {a} and {b}, and print their quotient",
    "Calculate the quotient of {a} and {b}",
    "Divide {a} by {b} and print the result",
    "Write a program to divide {a} by {b}",
    "Find the quotient of {a} divided by {b}",
    "Compute {a} divided by {b}",
    "What is {a} divided by {b}?",
    "Print the result of dividing {a} by {b}",
    "Show the quotient when you divide {a} by {b}",
    "Calculate {a} / {b}",
    "Divide {a} by {b} and display the answer",
    "What is the result of {a} divided by {b}?",
    "Find the division of {a} by {b}",
    "Get the quotient of {a} and {b}",
    "What do you get when you divide {a} by {b}?",
]

print(f"Templates: ADD={len(ADD_TEMPLATES)}, SUB={len(SUB_TEMPLATES)}, MUL={len(MUL_TEMPLATES)}, DIV={len(DIV_TEMPLATES)}")

In [ ]:
def generate_basic_add(a, b):
    return "10 INPUT A\n20 INPUT B\n30 PRINT A + B"

def generate_basic_sub(a, b):
    return "10 INPUT A\n20 INPUT B\n30 PRINT A - B"

def generate_basic_mul(a, b):
    return "10 INPUT A\n20 INPUT B\n30 PRINT A * B"

def generate_basic_div(a, b):
    return "10 INPUT A\n20 INPUT B\n30 PRINT A / B"

def generate_dataset(num_samples, seed=42):
    random.seed(seed)
    dataset = []
    ops = [
        ("add", ADD_TEMPLATES, generate_basic_add),
        ("sub", SUB_TEMPLATES, generate_basic_sub),
        ("mul", MUL_TEMPLATES, generate_basic_mul),
        ("div", DIV_TEMPLATES, generate_basic_div),
    ]
    per_op = num_samples // 4
    for op_name, templates, gen_func in ops:
        for _ in range(per_op):
            a = random.randint(0, 9)
            b = random.randint(0, 9)
            if op_name == "div" and b == 0:
                b = random.randint(1, 9)
            template = random.choice(templates)
            prompt = template.format(a=a, b=b)
            basic_code = gen_func(a, b)
            dataset.append({
                "prompt": prompt,
                "code": basic_code,
                "operation": op_name,
                "a": a,
                "b": b,
            })
    random.shuffle(dataset)
    return dataset

os.makedirs(config.data_dir, exist_ok=True)
dataset = generate_dataset(config.num_samples, config.seed)

with open(config.data_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print(f"Generated {len(dataset)} samples")
op_counts = {}
for item in dataset:
    op = item["operation"]
    op_counts[op] = op_counts.get(op, 0) + 1
for op, count in op_counts.items():
    print(f"  {op}: {count}")

print("\nSample data:")
for i in range(3):
    print(f"\n  Prompt: {dataset[i]['prompt']}")
    print(f"  Code:   {repr(dataset[i]['code'])}")

## Step 2: 分词器训练

使用字符级分词器 (CharTokenizer)：
- 特殊token: `<pad>`, `<eos>`, `<sep>`, `<unk>`
- 其余字符按ASCII字符表映射
- 词表大小约60-80，足够覆盖英文+BASIC关键字+数字+符号

In [ ]:
class CharTokenizer:
    def __init__(self):
        self.char_to_id = {}
        self.id_to_char = {}
        self.special_tokens = ["<pad>", "<eos>", "<sep>", "<unk>"]
        self.pad_id = 0
        self.eos_id = 1
        self.sep_id = 2
        self.unk_id = 3
        for i, token in enumerate(self.special_tokens):
            self.char_to_id[token] = i
            self.id_to_char[i] = token

    def train(self, texts):
        chars = set()
        for text in texts:
            chars.update(set(text))
        chars = sorted(chars)
        idx = len(self.special_tokens)
        for char in chars:
            if char not in self.char_to_id:
                self.char_to_id[char] = idx
                self.id_to_char[idx] = char
                idx += 1

    def encode(self, text):
        ids = []
        for char in text:
            if char in self.char_to_id:
                ids.append(self.char_to_id[char])
            else:
                ids.append(self.unk_id)
        return ids

    def decode(self, ids):
        chars = []
        for id_ in ids:
            if id_ in self.id_to_char:
                token = self.id_to_char[id_]
                if token == "<eos>":
                    break
                if token == "<pad>":
                    continue
                if token in self.special_tokens:
                    chars.append(" ")
                    continue
                chars.append(token)
            else:
                chars.append("?")
        return "".join(chars)

    def save(self, path):
        data = {
            "char_to_id": self.char_to_id,
            "special_tokens": self.special_tokens,
        }
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.char_to_id = {k: int(v) for k, v in data["char_to_id"].items()}
        self.id_to_char = {int(v): k for k, v in self.char_to_id.items()}
        self.special_tokens = data["special_tokens"]

    @property
    def get_vocab_size(self):
        return len(self.char_to_id)

all_texts = []
for item in dataset:
    all_texts.append(item["prompt"])
    all_texts.append(item["code"])

tokenizer = CharTokenizer()
tokenizer.train(all_texts)
tokenizer.save(config.tokenizer_path)

actual_vocab_size = tokenizer.get_vocab_size
config.vocab_size = actual_vocab_size

print(f"Vocab size: {actual_vocab_size}")
print(f"\nSample encoding:")
sample_text = "Input 3 and 4, and print their sum"
encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)
print(f"  Original: {sample_text}")
print(f"  Encoded:  {encoded}")
print(f"  Decoded:  {decoded}")
print(f"  Roundtrip OK: {sample_text == decoded}")

## Step 3: 模型构建

从零构建 GPT (Decoder-Only Transformer) 模型：
- **架构**: 6层 Transformer, 8头注意力, d_model=256, d_ff=1024
- **参数量**: ~15M
- **特性**: 因果注意力掩码, 位置编码, 权重共享(Embedding↔LM Head)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_model = d_model
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.pad_id = config.pad_id
        self.eos_id = config.eos_id
        self.sep_id = config.sep_id
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_id)
        self.pos_encoding = PositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)
        ])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.token_embedding.weight = self.head.weight
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model parameters: {n_params / 1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.padding_idx is not None:
                nn.init.zeros_(module.weight[module.padding_idx])
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        pad_mask = (input_ids != self.pad_id).unsqueeze(1).unsqueeze(2).expand(-1, 1, T, -1)
        mask = causal_mask & pad_mask
        tok_emb = self.token_embedding(input_ids)
        x = self.pos_encoding(tok_emb)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_len=128, temperature=0.8, top_k=50):
        self.eval()
        device = next(self.parameters()).device
        input_ids = input_ids.to(device)
        for _ in range(max_len):
            idx_cond = input_ids if input_ids.size(1) <= self.config.max_seq_len else input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if next_id.item() == self.eos_id:
                break
        return input_ids

print("Model class defined.")

In [ ]:
model = GPTModel(config)
model = model.to(device)

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel with {n_gpus} GPUs")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 4: 数据集与数据加载器

In [ ]:
class BasicDataset(Dataset):
    def __init__(self, data, tokenizer, max_seq_len, is_train=True):
        self.data = data
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.is_train = is_train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = item["prompt"]
        code = item["code"]
        prompt_ids = self.tokenizer.encode(prompt)
        code_ids = self.tokenizer.encode(code)
        input_ids = prompt_ids + [self.tokenizer.sep_id] + code_ids + [self.tokenizer.eos_id]
        labels = [-100] * len(prompt_ids) + [-100] + code_ids + [self.tokenizer.eos_id]
        if len(input_ids) > self.max_seq_len:
            input_ids = input_ids[:self.max_seq_len]
            labels = labels[:self.max_seq_len]
        pad_len = self.max_seq_len - len(input_ids)
        input_ids = input_ids + [self.tokenizer.pad_id] * pad_len
        labels = labels + [-100] * pad_len
        attention_mask = [1] * (self.max_seq_len - pad_len) + [0] * pad_len
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }

split_idx = int(len(dataset) * config.train_ratio)
train_data = dataset[:split_idx]
val_data = dataset[split_idx:]

train_dataset = BasicDataset(train_data, tokenizer, config.max_seq_len, is_train=True)
val_dataset = BasicDataset(val_data, tokenizer, config.max_seq_len, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print(f"Train: {len(train_data)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_data)} samples, {len(val_loader)} batches")

sample_batch = next(iter(train_loader))
print(f"\nBatch shape: input_ids={sample_batch['input_ids'].shape}, labels={sample_batch['labels'].shape}")

## Step 5: 模型训练

训练配置：
- 优化器: AdamW (lr=3e-4, weight_decay=0.01)
- 学习率调度: Cosine with Warmup (500步)
- 梯度裁剪: max_norm=1.0
- 损失函数: CrossEntropy (仅计算BASIC代码部分)

In [ ]:
def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

optimizer = AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
    betas=(0.9, 0.95),
)

total_steps = len(train_loader) * config.max_epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, config.warmup_steps, total_steps)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {config.warmup_steps}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
os.makedirs(config.model_dir, exist_ok=True)

best_val_loss = float("inf")
global_step = 0
train_losses = []
val_losses = []

print(f"Starting training for {config.max_epochs} epochs...")
print("=" * 70)

for epoch in range(config.max_epochs):
    model.train()
    total_loss = 0.0
    n_batches = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits, loss = model(input_ids, labels=labels)
        if n_gpus > 1:
            loss = loss.mean()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches += 1
        global_step += 1

        if global_step % 200 == 0:
            avg_loss = total_loss / n_batches
            lr = scheduler.get_last_lr()[0]
            print(f"  Step {global_step:5d} | Loss: {avg_loss:.4f} | LR: {lr:.6f}")

    avg_train_loss = total_loss / n_batches
    elapsed = time.time() - start_time
    train_losses.append(avg_train_loss)

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            logits, loss = model(input_ids, labels=labels)
            if n_gpus > 1:
                loss = loss.mean()
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch + 1:2d}/{config.max_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Time: {elapsed:.1f}s")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_model = model.module if n_gpus > 1 else model
        torch.save({
            "epoch": epoch,
            "model_state_dict": save_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": avg_val_loss,
            "config": {
                "vocab_size": config.vocab_size,
                "d_model": config.d_model,
                "n_heads": config.n_heads,
                "n_layers": config.n_layers,
                "d_ff": config.d_ff,
                "max_seq_len": config.max_seq_len,
                "dropout": config.dropout,
            },
        }, config.model_path)
        print(f"  -> Best model saved (val_loss: {avg_val_loss:.4f})")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=" * 70)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## Step 6: 推理验证

加载最佳模型，对测试提示语生成BASIC代码。

In [ ]:
checkpoint = torch.load(config.model_path, map_location="cpu")

saved_config = checkpoint.get("config", {})
inference_config = Config()
for k, v in saved_config.items():
    setattr(inference_config, k, v)

inference_model = GPTModel(inference_config)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model = inference_model.to(device)
inference_model.eval()

print(f"Model loaded from {config.model_path}")
print(f"Best val loss: {checkpoint['val_loss']:.4f}")

In [ ]:
def generate_basic(model, tokenizer, prompt, device,
                   max_len=128, temperature=0.8, top_k=50):
    prompt_ids = tokenizer.encode(prompt)
    input_ids = prompt_ids + [tokenizer.sep_id]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    output_ids = model.generate(
        input_tensor,
        max_len=max_len,
        temperature=temperature,
        top_k=top_k,
    )
    generated = output_ids[0].tolist()
    sep_pos = None
    for i, id_ in enumerate(generated):
        if id_ == tokenizer.sep_id:
            sep_pos = i
            break
    if sep_pos is not None:
        code_ids = generated[sep_pos + 1:]
    else:
        code_ids = generated[len(prompt_ids):]
    code_text = tokenizer.decode(code_ids)
    return code_text

test_prompts = [
    "Input 3 and 4, and print their sum",
    "Calculate the difference of 9 and 2",
    "Multiply 5 and 6 together and print the result",
    "Divide 8 by 4 and print the result",
    "Compute 7 plus 3",
    "What is 6 subtracted by 1?",
    "Find the product of 2 and 8",
    "What is 9 divided by 3?",
    "Add 0 and 5 together and print the result",
    "Calculate 8 - 7",
]

print("=" * 60)
print("BASIC Code Generation Results")
print("=" * 60)

for prompt in test_prompts:
    code = generate_basic(
        inference_model, tokenizer, prompt, device,
        max_len=config.inference_max_len,
        temperature=0.1,
        top_k=config.inference_top_k,
    )
    print(f"\nPrompt: {prompt}")
    print(f"BASIC Code:")
    for line in code.split(chr(10)):
        print(f"  {line}")
    print("-" * 40)

In [ ]:
correct = 0
total = len(test_prompts)
expected_ops = ["+", "-", "*", "/", "+", "-", "*", "/", "+", "-"]

for i, prompt in enumerate(test_prompts):
    code = generate_basic(
        inference_model, tokenizer, prompt, device,
        max_len=config.inference_max_len,
        temperature=0.1,
        top_k=config.inference_top_k,
    )
    has_input = "INPUT" in code
    has_print = "PRINT" in code
    has_op = expected_ops[i] in code
    is_correct = has_input and has_print and has_op
    correct += is_correct
    status = "PASS" if is_correct else "FAIL"
    print(f"[{status}] {prompt}")
    if not is_correct:
        print(f"       Generated: {repr(code)}")
        print(f"       Has INPUT: {has_input}, Has PRINT: {has_print}, Has '{expected_ops[i]}': {has_op}")

print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.1f}%)")

## Step 7: 自定义测试

可以输入自定义的提示语来测试模型。

In [ ]:
custom_prompts = [
    "Input 1 and 9, and print their sum",
    "Calculate the product of 7 and 3",
    "Subtract 4 from 8 and print the result",
    "What is 6 divided by 2?",
]

for prompt in custom_prompts:
    code = generate_basic(
        inference_model, tokenizer, prompt, device,
        max_len=config.inference_max_len,
        temperature=0.1,
        top_k=config.inference_top_k,
    )
    print(f"Prompt: {prompt}")
    print(f"Code:\n{code}")
    print("=" * 40)

## 清理资源

In [ ]:
del train_loader, val_loader, train_dataset, val_dataset
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print("Cleanup done.")